# 01 · Data preparation and TF‑IDF baseline

**Project:** *Do Fake News Classifiers Learn Truth or Tone?* A style-attack study of TF‑IDF + LR vs. DistilBERT.

This notebook covers **Stages 1–2** of the plan:

1. Load **WELFake** (72k articles) and check its label convention.
2. Clean the data: empty texts, very short texts, non-English articles, duplicates.
3. EDA, including **style cues** that differ between classes (wire-service datelines, `[VIDEO]` tags, ALL‑CAPS headlines). These motivate the whole project.
4. Make a stratified **train / val / test split by article ID**, saved to Google Drive for every later notebook.
5. Train the **TF‑IDF + Logistic Regression** baseline (C tuned on validation), evaluate it, and inspect its top features.

**How to run (Colab):** Runtime → *Run all*. A CPU runtime is enough for this notebook. Approve the Google Drive prompt when it appears.
All outputs go to `MyDrive/NLP_style_attack/`.

### ▶ Keep this tab awake (tap play once)

Phone browsers suspend background tabs, which can freeze the notebook's progress display or drop the
connection while long cells run. The player below loops a **near-silent** 5‑second sound. Tap ▶ once
after starting *Run all*; it keeps playing in your browser while the other cells run.
It does **not** extend Colab's own runtime limits, so keep the phone charging and Colab open.

In [ ]:
import io, wave, base64
import numpy as np
from IPython.display import HTML, display

_sr = 8000
_noise = (np.random.default_rng(0).standard_normal(_sr * 5) * 20).astype(np.int16)   # ~-64 dBFS: inaudible
_buf = io.BytesIO()
with wave.open(_buf, "wb") as _w:
    _w.setnchannels(1); _w.setsampwidth(2); _w.setframerate(_sr); _w.writeframes(_noise.tobytes())
display(HTML(f'<audio controls loop autoplay src="data:audio/wav;base64,{base64.b64encode(_buf.getvalue()).decode()}"></audio>'
             '<div style="font-size:12px;color:gray">Keep-alive player: tap ▶ if it is not already playing.</div>'))

## 0 · Configuration

In [ ]:
SEED = 42
SUBSAMPLE_N = 30_000        # articles kept for the project after cleaning (None = keep all)
SPLIT_FRACS = (0.70, 0.15, 0.15)
MIN_WORDS = 50              # drop articles shorter than this (too short to rewrite meaningfully)
MIN_STOPWORD_FRAC = 0.25    # English filter: share of tokens that are English stopwords
C_GRID = [0.1, 0.3, 1, 3, 10, 30]
PROJECT_DIRNAME = "NLP_style_attack"

## 1 · Setup

In [ ]:
import os, sys, re, json, hashlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = f"/content/drive/MyDrive/{PROJECT_DIRNAME}"
else:
    BASE_DIR = os.environ.get("FAKESTYLE_BASE_DIR", os.path.abspath(f"./{PROJECT_DIRNAME}"))

DIRS = {k: os.path.join(BASE_DIR, k) for k in ["data", "results", "models", "figures"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)
print("Outputs ->", BASE_DIR)

## 2 · Load WELFake

Sources, tried in order:
1. a cached copy on Drive from an earlier run,
2. the Hugging Face mirror `davanstrien/WELFake`,
3. the original Zenodo record (DOI 10.5281/zenodo.4561253).

In [ ]:
RAW_CACHE = os.path.join(DIRS["data"], "welfake_raw.parquet")
ZENODO_URL = "https://zenodo.org/api/records/4561253/files/WELFake_Dataset.csv/content"

def load_welfake():
    local_csv = os.environ.get("FAKESTYLE_LOCAL_CSV")       # for offline testing only
    if local_csv:
        return pd.read_csv(local_csv), f"local file {local_csv}"
    if os.path.exists(RAW_CACHE):
        return pd.read_parquet(RAW_CACHE), "Drive cache"
    try:
        from datasets import load_dataset
        df = load_dataset("davanstrien/WELFake", split="train").to_pandas()
        source = "Hugging Face (davanstrien/WELFake)"
    except Exception as e:
        print("Hugging Face load failed:", repr(e)[:200], "-> trying Zenodo")
        df = pd.read_csv(ZENODO_URL)
        source = "Zenodo"
    return df, source

raw, source = load_welfake()
raw = raw.rename(columns=str.lower)
raw = raw[["title", "text", "label"]].copy()
raw["label"] = raw["label"].astype(int)
if not os.path.exists(RAW_CACHE) and not os.environ.get("FAKESTYLE_LOCAL_CSV"):
    raw.to_parquet(RAW_CACHE)
raw = raw.reset_index(drop=True)
raw.insert(0, "id", raw.index.astype(int))   # stable article ID used everywhere downstream
print(f"Loaded {len(raw):,} rows from {source}")
raw.head()

### 2.1 · Which label means *fake*?

The WELFake documentation says `0 = fake, 1 = real`, but the published data appears to use the
**opposite** convention: Reuters wire stories are labelled `0`. Rather than trust either, we infer
the mapping from the data: the label whose articles carry Reuters datelines far more often is *real*.

In [ ]:
reuters_re = re.compile(r"\(Reuters\)")
has_reuters = raw["text"].fillna("").str.contains(reuters_re)
reuters_share = has_reuters.groupby(raw["label"]).mean()
print("Share of articles containing '(Reuters)' by raw label:\n", reuters_share.round(4))

REAL_LABEL = int(reuters_share.idxmax())
assert reuters_share.max() > 5 * max(reuters_share.min(), 1e-6), "Label mapping is ambiguous - inspect manually"
raw["fake"] = (raw["label"] != REAL_LABEL).astype(int)
print(f"\n=> raw label {REAL_LABEL} = REAL, raw label {1 - REAL_LABEL} = FAKE.  Column `fake`: 1 = fake, 0 = real")

## 3 · Cleaning

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

log = [("loaded", len(raw))]
df = raw.copy()
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

df = df[df["text"].str.len() > 0];                 log.append(("non-empty text", len(df)))
df["n_words"] = df["text"].str.split().str.len()
df = df[df["n_words"] >= MIN_WORDS];               log.append((f">= {MIN_WORDS} words", len(df)))

token_re = re.compile(r"[a-z']+")
def stopword_frac(s):
    toks = token_re.findall(s[:3000].lower())
    return sum(t in ENGLISH_STOP_WORDS for t in toks) / max(len(toks), 1)
df["sw_frac"] = df["text"].map(stopword_frac)
df = df[df["sw_frac"] >= MIN_STOPWORD_FRAC];        log.append(("English", len(df)))

# Duplicates: compare a normalised form of the text (lowercase, letters/digits only).
norm = df["text"].str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
df["text_hash"] = norm.map(lambda s: hashlib.md5(s.encode()).hexdigest())
n_labels = df.groupby("text_hash")["fake"].transform("nunique")
conflicting = (n_labels > 1).sum()
df = df[n_labels == 1]                              # same text with both labels -> unreliable, drop all copies
log.append(("no conflicting-label duplicates", len(df)))
df = df.drop_duplicates("text_hash", keep="first"); log.append(("deduplicated", len(df)))

print(pd.DataFrame(log, columns=["step", "rows"]).to_string(index=False))
print(f"\n(rows whose text appears with BOTH labels and were dropped: {conflicting:,})")

## 4 · Exploratory analysis

In [ ]:
balance = df["fake"].map({0: "real", 1: "fake"}).value_counts()
print("Class balance after cleaning:\n", balance, "\n")
print(df.groupby(df["fake"].map({0: "real", 1: "fake"}))["n_words"].describe().round(0))

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.logspace(np.log10(MIN_WORDS), np.log10(df["n_words"].max() + 1), 50)
for lab, name in [(0, "real"), (1, "fake")]:
    ax.hist(df.loc[df["fake"] == lab, "n_words"], bins=bins, alpha=0.55, label=name)
ax.set_xscale("log"); ax.set_xlabel("article length (words, log scale)"); ax.set_ylabel("articles")
ax.set_title("Article length by class"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "01_length_by_class.png"), dpi=150)
plt.show()

### 4.1 · Style cues by class

If simple **surface cues** separate the classes, a classifier can score well without judging content at all.
This table is the core motivation for the style-attack experiments.

In [ ]:
title, text = df["title"], df["text"]
letters = title.str.replace(r"[^A-Za-z]", "", regex=True)
upper_frac = letters.str.count(r"[A-Z]") / letters.str.len().clip(lower=1)

CUES = {
    "wire dateline 'CITY (Reuters) -'": text.str.contains(r"^\s*[A-Z][\w .,/'-]{0,60}\(Reuters\)\s*[-–—]", regex=True),
    "title ends '- The New York Times'": title.str.contains(r"- The New York Times\s*$", regex=True),
    "title ends '- Breitbart'": title.str.contains(r"- Breitbart\s*$", regex=True),
    "[VIDEO]/(VIDEO)/[IMAGES] tag in title": title.str.contains(r"[\[(]\s*(?:video|images?|watch)\s*[\])]", case=False, regex=True),
    "'!' in title": title.str.contains("!", regex=False),
    "title mostly UPPERCASE (>60%)": (upper_frac > 0.6) & (letters.str.len() >= 10),
    "URL in text": text.str.contains(r"https?://|pic\.twitter\.com", regex=True),
    "empty title": title.str.len() == 0,
}
cue_df = pd.DataFrame({name: s.groupby(df["fake"]).mean() for name, s in CUES.items()}).T
cue_df.columns = ["% of real", "% of fake"]
cue_df = (cue_df * 100).round(1)
cue_df["ratio (larger/smaller)"] = (cue_df.max(axis=1) / cue_df.min(axis=1).clip(lower=0.1)).where(cue_df.max(axis=1) > 0).round(1)
cue_df.to_csv(os.path.join(DIRS["results"], "01_style_cues.csv"))
cue_df

### 4.2 · Examples

In [ ]:
for lab, name in [(0, "REAL"), (1, "FAKE")]:
    print(f"===== {name} =====")
    for _, r in df[df["fake"] == lab].sample(3, random_state=SEED).iterrows():
        print(f"[{r['id']}] {r['title'][:110]}\n    {r['text'][:260]}...\n")

## 5 · Subsample and split by article ID

We keep `SUBSAMPLE_N` articles (stratified), so DistilBERT training and the LLM rewrites stay manageable,
then split 70 / 15 / 15 with stratification. **Every rewrite made later inherits its source article's split**,
so no article leaks between train and test.

In [ ]:
from sklearn.model_selection import train_test_split

pool = df if SUBSAMPLE_N is None or SUBSAMPLE_N >= len(df) else \
    train_test_split(df, train_size=SUBSAMPLE_N, stratify=df["fake"], random_state=SEED)[0]

tr_frac, va_frac, te_frac = SPLIT_FRACS
train_df, rest = train_test_split(pool, train_size=tr_frac, stratify=pool["fake"], random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=te_frac / (va_frac + te_frac), stratify=rest["fake"], random_state=SEED)

KEEP = ["id", "title", "text", "fake", "n_words"]
splits = {"train": train_df[KEEP], "val": val_df[KEEP], "test": test_df[KEEP]}
for name, part in splits.items():
    part.reset_index(drop=True).to_parquet(os.path.join(DIRS["data"], f"{name}.parquet"))

assert not (set(train_df.id) & set(val_df.id)) and not (set(train_df.id) & set(test_df.id)) and not (set(val_df.id) & set(test_df.id))
print(pd.DataFrame({n: {"articles": len(p), "% fake": round(100 * p["fake"].mean(), 1)} for n, p in splits.items()}).T)
print("\nSaved to", DIRS["data"])

## 6 · Baseline: TF‑IDF + Logistic Regression

Input = `title + text`. The vectoriser is fitted on **train only**; `C` is chosen on **validation**; the test set is used once.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

def model_input(d):
    return (d["title"] + "\n\n" + d["text"]).tolist()

X_tr_txt, X_va_txt, X_te_txt = map(model_input, (train_df, val_df, test_df))
y_tr, y_va, y_te = train_df["fake"].values, val_df["fake"].values, test_df["fake"].values

t0 = time.time()
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                             max_features=300_000, strip_accents="unicode")
X_tr = vectorizer.fit_transform(X_tr_txt)
X_va, X_te = vectorizer.transform(X_va_txt), vectorizer.transform(X_te_txt)
print(f"TF-IDF: {X_tr.shape[1]:,} features, fitted in {time.time() - t0:.0f}s")

val_scores = {}
for C in C_GRID:
    clf = LogisticRegression(C=C, max_iter=3000, solver="liblinear", random_state=SEED).fit(X_tr, y_tr)
    val_scores[C] = f1_score(y_va, clf.predict(X_va), average="macro")
    print(f"  C={C:<5} val macro-F1 = {val_scores[C]:.4f}")
BEST_C = max(val_scores, key=val_scores.get)
clf = LogisticRegression(C=BEST_C, max_iter=3000, solver="liblinear", random_state=SEED).fit(X_tr, y_tr)
print("Best C =", BEST_C)

In [ ]:
def evaluate(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "precision_fake": precision_score(y_true, y_pred, pos_label=1),
        "recall_fake": recall_score(y_true, y_pred, pos_label=1),
        "precision_real": precision_score(y_true, y_pred, pos_label=0),
        "recall_real": recall_score(y_true, y_pred, pos_label=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "n": int(len(y_true)),
    }

results = {}
for name, X, y in [("val", X_va, y_va), ("test", X_te, y_te)]:
    prob = clf.predict_proba(X)[:, 1]
    results[name] = evaluate(y, (prob >= 0.5).astype(int), prob)
res_df = pd.DataFrame(results).T.round(4)
print(res_df)

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, clf.predict(X_te)), display_labels=["real", "fake"]).plot(ax=ax, colorbar=False)
ax.set_title("TF-IDF + LR · test"); fig.tight_layout()
fig.savefig(os.path.join(DIRS["figures"], "01_tfidf_confusion_test.png"), dpi=150); plt.show()

### 6.1 · What does the baseline rely on?

The highest-weighted n‑grams for each class. Watch for **source and style artefacts** (agency names,
datelines, weekday phrasing like *"said on Tuesday"*, `video`, `featured image`) rather than content.
These are what the style attack in notebook 03 is designed to exploit.

In [ ]:
TOP_K = 30
feats = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]
order = np.argsort(coefs)
top = pd.DataFrame({
    "→ REAL (n-gram)": feats[order[:TOP_K]], "weight_real": coefs[order[:TOP_K]].round(2),
    "→ FAKE (n-gram)": feats[order[::-1][:TOP_K]], "weight_fake": coefs[order[::-1][:TOP_K]].round(2),
})
top.to_csv(os.path.join(DIRS["results"], "01_tfidf_top_features.csv"), index=False)
top

## 7 · Save model and results

In [ ]:
import joblib, sklearn
joblib.dump({"vectorizer": vectorizer, "clf": clf}, os.path.join(DIRS["models"], "tfidf_lr.joblib"))

summary = {
    "notebook": "01_data_and_baseline",
    "data_source": source,
    "real_label_in_raw_data": REAL_LABEL,
    "cleaning_log": dict(log),
    "conflicting_label_duplicates_dropped": int(conflicting),
    "split_sizes": {n: len(p) for n, p in splits.items()},
    "config": {"SEED": SEED, "SUBSAMPLE_N": SUBSAMPLE_N, "MIN_WORDS": MIN_WORDS,
               "MIN_STOPWORD_FRAC": MIN_STOPWORD_FRAC, "best_C": BEST_C,
               "tfidf_features": int(X_tr.shape[1]), "sklearn": sklearn.__version__},
    "val_macro_f1_by_C": {str(k): round(v, 4) for k, v in val_scores.items()},
    "metrics": {k: {m: round(float(v), 4) for m, v in d.items()} for k, d in results.items()},
    "style_cues_pct": cue_df.drop(columns="ratio (larger/smaller)").to_dict(orient="index"),
}
with open(os.path.join(DIRS["results"], "01_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

**Done.** `MyDrive/NLP_style_attack/` now contains:

| Path | Contents |
|---|---|
| `data/{train,val,test}.parquet` | Cleaned splits (`id, title, text, fake, n_words`); used by every later notebook |
| `results/01_summary.json` | Cleaning log, split sizes, metrics, style-cue rates |
| `results/01_style_cues.csv`, `results/01_tfidf_top_features.csv` | EDA and feature tables |
| `models/tfidf_lr.joblib` | Fitted vectoriser + classifier, reused for the style attack |
| `figures/` | Length histogram, confusion matrix |

Next: **02 · DistilBERT fine-tuning** (needs a GPU runtime).